# выбрал датасет с росстата: https://rosstat.gov.ru/opendata/7708234640-population

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/lab07-data.csv", encoding='cp1251')

df.head()
df.describe()

,number,year,kode,total,urban,rural,Unnamed: 8,Unnamed: 9,Unnamed: 10
count,25589.000000,25589.0,2.552100e+04,2.552100e+04,2.551100e+04,2.551500e+04,0.0,0.0,0.0
mean,12795.000000,2014.0,4.877020e+13,1.909141e+04,1.476690e+04,4.331317e+03,NaN,NaN,NaN
std,7387.052355,0.0,1.694830e+14,1.536197e+05,1.313758e+05,3.521085e+04,NaN,NaN,NaN
min,1.000000,2014.0,1.000000e+08,0.000000e+00,0.000000e+00,0.000000e+00,NaN,NaN,NaN
25%,6398.000000,2014.0,2.763400e+09,7.930000e+02,0.000000e+00,4.210000e+02,NaN,NaN,NaN
50%,12795.000000,2014.0,5.763642e+09,1.624000e+03,0.000000e+00,9.820000e+02,NaN,NaN,NaN
75%,19192.000000,2014.0,8.465000e+09,5.911000e+03,0.000000e+00,2.154000e+03,NaN,NaN,NaN
max,25589.000000,2014.0,9.970100e+14,1.210826e+07,1.197166e+07,2.491490e+06,NaN,NaN,NaN


In [2]:
df.isnull().sum()

number                0
year                  0
kode                 68
region                0
municipalities        0
total                68
urban                78
rural                74
Unnamed: 8        25589
Unnamed: 9        25589
Unnamed: 10       25589
dtype: int64

In [3]:
df = df.dropna(axis=1, how='all')
df.columns

Index(['number', 'year', 'kode', 'region', 'municipalities', 'total', 'urban',
       'rural'],
      dtype='str')

In [4]:
df

,number,year,kode,region,municipalities,total,urban,rural
0,1,2014,7.900000e+09,Республика Адыгея,Муниципальные образования Республики Адыгеи,446406.0,209929.0,236477.0
1,2,2014,7.970100e+09,Республика Адыгея,Городской округ 'Город Майкоп',167620.0,144544.0,23076.0
2,3,2014,7.970100e+14,Республика Адыгея,г. Майкоп,144544.0,144544.0,0.0
3,4,2014,7.970300e+09,Республика Адыгея,Городской округ 'Город Адыгейск',14935.0,12481.0,2454.0
4,5,2014,7.970300e+14,Республика Адыгея,г. Адыгейск,12481.0,12481.0,0.0
...,...,...,...,...,...,...,...,...
25584,25585,2014,7.763342e+09,Чукотский автономный округ,Сельское поселение Лаврентия,1388.0,0.0,1388.0
25585,25586,2014,7.763342e+09,Чукотский автономный округ,Сельское поселение Лорино,1160.0,0.0,1160.0
25586,25587,2014,7.763343e+09,Чукотский автономный округ,Сельское поселение Нешкан,675.0,0.0,675.0
25587,25588,2014,7.763344e+09,Чукотский автономный округ,Сельское поселение Уэлен,697.0,0.0,697.0


In [5]:
df['target'] = pd.qcut(df['total'], q=3, labels=[0,1,2])
df = df.dropna()
X = df.drop(['target', 'total'], axis=1)
y = df['target']


In [6]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = X.select_dtypes(include=['object', 'string']).columns

encoder = OrdinalEncoder()
X[cat_cols] = encoder.fit_transform(X[cat_cols])

X.head()

,number,year,kode,region,municipalities,urban,rural
0,1,2014,7.900000e+09,44.0,6614.0,209929.0,236477.0
1,2,2014,7.970100e+09,44.0,3004.0,144544.0,23076.0
2,3,2014,7.970100e+14,44.0,19632.0,144544.0,0.0
3,4,2014,7.970300e+09,44.0,2991.0,12481.0,2454.0
4,5,2014,7.970300e+14,44.0,19103.0,12481.0,0.0


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

lr_params = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["lbfgs"]
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    lr_params,
    cv=5,
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

lr_best = lr_grid.best_estimator_

lr_pred = lr_best.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)

lr_acc

0.3254263869829445

In [9]:
from sklearn.tree import DecisionTreeClassifier

tree_params = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10]
}

tree_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    tree_params,
    cv=5,
    n_jobs=-1
)

tree_grid.fit(X_train, y_train)

tree_best = tree_grid.best_estimator_

tree_pred = tree_best.predict(X_test)
tree_acc = accuracy_score(y_test, tree_pred)

tree_acc

0.9996079200156832

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_

rf_pred = rf_best.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

rf_acc

0.999019800039208

In [11]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

lsvc_params = {
    "C": [0.01, 0.1, 1, 10],
    "loss": ["hinge", "squared_hinge"]
}

lsvc_grid = GridSearchCV(
    LinearSVC(max_iter=5000),
    lsvc_params,
    cv=5,
    n_jobs=-1
)

lsvc_grid.fit(X_train, y_train)

lsvc_best = lsvc_grid.best_estimator_
lsvc_grid.best_params_

lsvc_pred = lsvc_best.predict(X_test)
lsvc_acc = accuracy_score(y_test, lsvc_pred)

lsvc_acc

C:\Users\Daniil\PycharmProjects\CS351\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.33875710644971574

In [12]:
from sklearn.neighbors import KNeighborsClassifier

knn_params = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_params,
    cv=5,
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

knn_best = knn_grid.best_estimator_

knn_pred = knn_best.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)

knn_acc

0.6710448931582043

In [14]:
from sklearn.ensemble import VotingClassifier

voting_best = VotingClassifier(
    estimators=[
        ("lr", lr_best),
        ("tree", tree_best),
        ("rf", rf_best),
    ],
    voting="soft"
)

voting_best.fit(X_train, y_train)

voting_pred = voting_best.predict(X_test)
voting_acc = accuracy_score(y_test, voting_pred)

voting_acc

0.9996079200156832